# MELD text experiments on a T4

Run the sections in order: dataset auditing, TF-IDF + Logistic Regression,
DeBERTa-v3-base training, checkpoint reuse, dev-selected ensembles, held-out
evaluation, learning curves, class-wise reports and token-visible inference.

Checkpoint and ensemble choices use dev only. Test is evaluated after those
choices are frozen. Previously inspected benchmark results must be disclosed
when starting follow-up experiments.

Before committing, run python scripts/build_colab_bundle.py locally and
upload dist/mer_experiments_colab.zip in the first cell. After committing,
SOURCE='github' can load the same project code.


In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile
SOURCE = "upload"  # "github" only after committing the patch
ARCHIVE = Path("/content/mer_experiments_colab.zip")
if SOURCE == "upload":
    if not ARCHIVE.exists():
        from google.colab import files
        uploaded = files.upload()
        name = next((n for n in uploaded if n.endswith(".zip")), None)
        if name is None:
            raise ValueError("Upload dist/mer_experiments_colab.zip")
        ARCHIVE.write_bytes(uploaded[name])
    ROOT = Path("/content") / ("mer-experiments-" + hashlib.sha256(ARCHIVE.read_bytes()).hexdigest()[:10])
    ROOT.mkdir(exist_ok=True)
    with zipfile.ZipFile(ARCHIVE) as archive:
        if sum(i.file_size for i in archive.infolist()) > 20 * 1024**2:
            raise ValueError("Unexpectedly large source archive")
        for info in archive.infolist():
            target = (ROOT / info.filename).resolve()
            if not target.is_relative_to(ROOT.resolve()) or info.filename.startswith(("/", "\\")):
                raise ValueError("Unsafe archive member")
        archive.extractall(ROOT)
    manifest = json.loads((ROOT / "source_manifest.json").read_text())
    for name, expected in manifest["files"].items():
        if hashlib.sha256((ROOT / name).read_bytes()).hexdigest() != expected:
            raise ValueError(f"Source hash mismatch: {name}")
else:
    ROOT = Path("/content/mer-experiments-github")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "https://github.com/NurimanB/Multimodal-Emotion-Recognition.git", str(ROOT)], check=True)
    if not (ROOT / "requirements-experiments.txt").exists():
        raise RuntimeError("The experiment patch is not on this GitHub revision. Use SOURCE='upload'.")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("Source:", ROOT)

## 1. Install the recorded software versions

This preserves Colab's CUDA-enabled PyTorch. If the cell requests a restart, restart the **session** and rerun the setup cells; uploaded files remain on /content. No GitHub token is needed for ZIP upload.

In [ ]:
import importlib.metadata
loaded = {name: getattr(sys.modules[name], "__version__", None)
          for name in ["numpy", "pandas", "transformers"] if name in sys.modules}
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-experiments.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
changed = [name for name, version in loaded.items() if version != importlib.metadata.version(name)]
if changed:
    raise RuntimeError(f"Restart the Colab session and rerun setup: preloaded packages changed: {changed}")
import torch, transformers, pandas as pd
print("torch:", torch.__version__, "transformers:", transformers.__version__)
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime before training")
print("GPU:", torch.cuda.get_device_name(0), "| GiB:", torch.cuda.get_device_properties(0).total_memory / 2**30)

## 2. Drive paths, saved artifacts and predeclared experiments

Default: one seed on T4, maximum 6 epochs with dev early stopping. For variability, declare [42, 123, 2025] **before** running the training cell. This triples the training work. Full checkpoints are local by default to avoid filling Drive; finished best models and logs are saved to Drive. An interrupted unfinished local run cannot survive deletion of its Colab VM.

Set legacy checkpoint/config/state paths to your actual Drive artifacts. The existing RoBERTa checkpoint can be evaluated without retraining. Old probability directories must contain full, sorted MELD dev/test arrays. Enabling ALLOW_LEGACY_ORDER accepts the old pipeline's missing ID metadata only after checking the full label sequence.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/diploma-mer")
TAG = "v1"  # change for a new development experiment, not after inspecting test
from mer.evaluation.paths import locate_run
BASE = locate_run(DRIVE, "v1") if TAG == "v1" else DRIVE / "experiments" / TAG
DATA = DRIVE / "data/meld"
BASE.mkdir(parents=True, exist_ok=True)
SEEDS = [42]  # optionally [42, 123, 2025], declared before training
TRAIN_NEW_MODEL = True
RUN_SMOKE = False
RUN_FINAL_TEST = True  # used only AFTER the dev selection/freeze section
ROBERTA_CHECKPOINT = DRIVE / "outputs/text_roberta_large_ctx3/saved_model"
ROBERTA_CONFIG = DRIVE / "outputs/text_roberta_large_ctx3/config.json"
ROBERTA_STATE = ROBERTA_CHECKPOINT / "trainer_state.json"
LEGACY_PROB_RUNS = []  # e.g. [DRIVE/"outputs/your_xlmr_run"]; maximum 4 ensemble members total
ALLOW_LEGACY_ORDER = False
def run_module(module, *arguments):
    command = [sys.executable, "-m", module, *map(str, arguments)]
    print("Running:", " ".join(command))
    subprocess.run(command, cwd=ROOT, check=True)
from mer.evaluation.artifacts import write_json
if (ROOT / "source_manifest.json").exists() and not (BASE / "source_manifest.json").exists():
    (BASE / "source_manifest.json").write_bytes((ROOT / "source_manifest.json").read_bytes())

## 3. Dataset evidence and leakage screening

Keep the official train/dev/test splits. This reports class counts, empty text, duplicate sample IDs, exact cross-split overlap, and a character-TF-IDF near-duplicate screen. Common dialogue phrases can repeat: disclose this instead of claiming zero leakage. Near matches need manual review; do not resplit the benchmark to improve a reported score.

In [ ]:
from mer.data.meld import download_meld
download_meld(DATA)
run_module("mer.evaluation.audit", "--data-dir", DATA, "--output-dir", BASE/"dataset", "--near-duplicates")
display(pd.read_csv(BASE/"dataset/class_distribution.csv").pivot(index="emotion", columns="split", values="count"))
print((BASE/"dataset/dataset_audit.json").read_text())

## 4. Meaningful baseline — train-only TF-IDF + Logistic Regression

Same speaker/previous-turn input policy and official evaluation split. Vocabulary/IDF fit only on train. C and class weighting are selected by dev weighted F1. This cell does not access test performance.

In [ ]:
BASELINE = BASE / "tfidf"
if not (BASELINE/"baseline.joblib").exists():
    run_module("mer.evaluation.baseline", "--data-dir", DATA, "--output-dir", BASELINE, "--n-context", 3)
print(json.loads((BASELINE/"results.json").read_text())["dev"])
display(pd.read_csv(BASELINE/"dev_search.csv"))

## 5. Train the third architecture — DeBERTa-v3-base

Encoder → pooled representation → seven-class head → softmax; all parameters fine-tuned. T4 preset: batch 4, accumulation 4, max length 256, gradient checkpointing, automatic CUDA mixed precision, past context only. Select best checkpoint by dev weighted F1. This is an English MELD experiment, not multilingual validation.

The optional smoke run uses a small subset solely to check execution. Its scores must not appear in the comparison table. Completed runs with identical configs are reused. Do not overwrite a run with a different configuration.

In [ ]:
import yaml
from mer.text.train import DEFAULTS
CONFIG = ROOT / "configs/text_deberta_v3_base.yaml"
if RUN_SMOKE:
    run_module("mer.text.train", "--config", ROOT/"configs/smoke_deberta.yaml",
               f"data_dir={DATA}", f"output_dir={BASE/'smoke'}", "checkpoint_dir=/content/checkpoints/experiment_smoke")
NEW_RUNS = []
for seed in SEEDS:
    output = BASE / f"deberta_s{seed}"
    checkpoint = Path("/content/checkpoints") / TAG / f"deberta_s{seed}"
    expected = {**DEFAULTS, **yaml.safe_load(CONFIG.read_text()), "data_dir": str(DATA),
                "output_dir": str(output), "checkpoint_dir": str(checkpoint), "seed": seed}
    if (output/"results.json").exists() and (output/"best_model/config.json").exists():
        actual = json.loads((output/"config.json").read_text())
        storage_keys = {"data_dir", "output_dir", "checkpoint_dir", "resume_from_checkpoint"}
        if ({k: v for k, v in actual.items() if k not in storage_keys}
                != {k: v for k, v in expected.items() if k not in storage_keys}):
            raise RuntimeError(f"Configuration changed for {output}; use a new TAG")
        print("Reusing completed run:", output)
    elif TRAIN_NEW_MODEL:
        run_module("mer.text.train", "--config", CONFIG, f"data_dir={DATA}",
                   f"output_dir={output}", f"checkpoint_dir={checkpoint}", f"seed={seed}")
    else:
        raise FileNotFoundError(f"Train first or supply a completed run: {output}")
    NEW_RUNS.append(output)
display(pd.DataFrame([{"seed": s, "dev_weighted_f1": json.loads((r/"results.json").read_text())["dev"]["weighted_f1"]}
                     for s, r in zip(SEEDS, NEW_RUNS)]))

## 6. Reuse the existing RoBERTa checkpoint — dev only

No retraining. A missing run config is an evidence gap: supply the actual training config rather than guessing context/loss settings. The saved checkpoint architecture/label map is checked during loading. Legacy JSON filenames alone do not prove checkpoint identity.

In [ ]:
ROBERTA_RUN = BASE / "roberta_recheck"
if ROBERTA_CHECKPOINT.is_dir() and ROBERTA_CONFIG.is_file():
    if not (ROBERTA_RUN/"dev_probs.npy").exists():
        run_module("mer.evaluation.inference", "--checkpoint", ROBERTA_CHECKPOINT, "--config", ROBERTA_CONFIG,
                   "--data-dir", DATA, "--output-dir", ROBERTA_RUN, "--splits", "dev", "--batch-size", 8)
    print("RoBERTa dev weighted F1:", json.loads((ROBERTA_RUN/"results.json").read_text())["dev"]["weighted_f1"])
else:
    print("Legacy RoBERTa evaluation skipped: set its actual checkpoint and run-config paths.")
print("Trainer state available:", ROBERTA_STATE.is_file())

## 7. Freeze all choices using dev — no test arrays loaded here

The best new seed is selected on dev. Report all predeclared seeds afterward, including mean/std when there are at least two. Ensemble grid: quarter-step nonnegative weights plus equal weights, neutral log-score bias from a fixed five-value grid, chosen by dev F1. The reported old 0.664 is not inserted as a measured value.

In [ ]:
BEST_NEW = max(NEW_RUNS, key=lambda r: json.loads((r/"results.json").read_text())["dev"]["weighted_f1"])
ENSEMBLE_RUNS = ([ROBERTA_RUN] if (ROBERTA_RUN/"dev_probs.npy").exists() else []) + [BEST_NEW] + list(map(Path, LEGACY_PROB_RUNS))
if not 1 <= len(ENSEMBLE_RUNS) <= 4:
    raise ValueError("Select 1-4 ensemble member runs")
ENSEMBLE = BASE / "ensemble"
freeze_file = BASE / "frozen_protocol.json"
def artifact_hashes():
    paths = [BASELINE/"config.json", BASELINE/"dev_probs.npy"]
    for directory in set(NEW_RUNS + ENSEMBLE_RUNS):
        paths += [directory/"config.json", directory/"dev_probs.npy"]
    return {str(p): hashlib.sha256(p.read_bytes()).hexdigest() for p in paths if p.exists()}
def selection_hash():
    path = ENSEMBLE/"selection.json"
    return hashlib.sha256(path.read_bytes()).hexdigest() if path.exists() else None
current = {"seeds": SEEDS, "best_new": str(BEST_NEW), "ensemble_runs": list(map(str, ENSEMBLE_RUNS)),
           "artifact_hashes": artifact_hashes(), "selection_split": "dev",
           "ensemble_selection_sha256": selection_hash()}
if freeze_file.exists():
    if json.loads(freeze_file.read_text()) != current:
        raise RuntimeError("Artifacts differ from frozen protocol. Do not reuse these test results for new tuning.")
else:
    arguments = list(ENSEMBLE_RUNS) + ["--data-dir", DATA, "--output-dir", ENSEMBLE]
    if ALLOW_LEGACY_ORDER:
        arguments += ["--allow-legacy-order"]
    run_module("mer.evaluation.ensemble", *arguments)
    current["ensemble_selection_sha256"] = selection_hash()
    write_json(freeze_file, current)
print("Frozen model:", BEST_NEW)
print((ENSEMBLE/"selection.json").read_text())

## 8. Final held-out evaluation — after freeze

This evaluates each predeclared new seed and the baseline. Existing test artifacts are reused on reruns. After this section, use errors for error analysis; further tuning needs a new clearly documented development protocol, not a claim that this test remains unseen.

In [ ]:
if RUN_FINAL_TEST:
    if (json.loads(freeze_file.read_text()) != current or artifact_hashes() != current["artifact_hashes"]
            or selection_hash() != current["ensemble_selection_sha256"]):
        raise RuntimeError("Frozen artifacts changed")
    from mer.evaluation.baseline import evaluate_saved
    if "test" not in json.loads((BASELINE/"results.json").read_text()):
        evaluate_saved(BASELINE, DATA)
    for directory in NEW_RUNS:
        if "test" not in json.loads((directory/"results.json").read_text()):
            run_module("mer.evaluation.inference", "--checkpoint", directory/"best_model", "--config", directory/"config.json",
                       "--data-dir", DATA, "--output-dir", directory, "--splits", "test", "--batch-size", 8)
    if ROBERTA_RUN in ENSEMBLE_RUNS and "test" not in json.loads((ROBERTA_RUN/"results.json").read_text()):
        run_module("mer.evaluation.inference", "--checkpoint", ROBERTA_CHECKPOINT, "--config", ROBERTA_CONFIG,
                   "--data-dir", DATA, "--output-dir", ROBERTA_RUN, "--splits", "test", "--batch-size", 8)
    if "test" not in json.loads((ENSEMBLE/"results.json").read_text()):
        run_module("mer.evaluation.ensemble", "--data-dir", DATA, "--output-dir", ENSEMBLE, "--evaluate-frozen")
else:
    print("Test evaluation disabled. Subsequent reports will explicitly mark test as not measured.")

## 9. Confusion matrices, per-class P/R/F1, curves, correct/incorrect examples

Each report includes CSVs and PNGs. Inspect the five correct and five incorrect examples and fill in human_error_hypothesis/human_notes. Consider missing prosody, sarcasm, ambiguous labels, short utterances and class imbalance as hypotheses; do not assert them without inspecting each example. Determine the actual weakest classes from the computed report.

In [ ]:
from mer.evaluation.report import make_report
REPORT_RUNS = [BASELINE] + NEW_RUNS + ([ROBERTA_RUN] if (ROBERTA_RUN/"results.json").exists() else []) + [ENSEMBLE]
for directory in REPORT_RUNS:
    state = ROBERTA_STATE if directory == ROBERTA_RUN and ROBERTA_STATE.is_file() else None
    make_report(directory, trainer_state=state)
# Historical JSONs can provide matrices/tables without weights; no per-example results are fabricated.
for filename, title in [("RoBERTa-large_result.json", "Historical RoBERTa-large"),
                        ("XLM-RoBERTa-large_result.json", "Historical XLM-RoBERTa-large")]:
    make_report(ROOT/"results", output_dir=BASE/"historical"/filename.removesuffix(".json"),
                results_file=ROOT/"results"/filename, title=title,
                trainer_state=ROBERTA_STATE if filename.startswith("RoBERTa-") and ROBERTA_STATE.is_file() else None)
from mer.evaluation.summary import summarize
display(summarize(REPORT_RUNS, BASE/"comparison"))
from IPython.display import Image, display
for filename in ["learning_curves.png", "test_confusion_matrix.png", "test_confusion_normalized.png"]:
    path = BEST_NEW/"report"/filename
    if path.exists():
        display(Image(filename=str(path)))
examples = BEST_NEW/"report/test_review_examples.csv"
if examples.exists():
    display(pd.read_csv(examples))
print("All evidence:", BASE)

## 10. Live demo — text → cleaning → tokens/IDs/mask → emotion + confidence

Runs your saved best checkpoint. Context is optional; provide previous turns with speaker names to match training. Unknown speakers, non-English text, sarcasm and out-of-domain text can fail. Confidence is an uncalibrated softmax score.

In [ ]:
from mer.evaluation.inference import Predictor, launch_widget
DEMO = Predictor(BEST_NEW/"best_model", BEST_NEW/"config.json")
launch_widget(DEMO)

## 11. Export experiment artifacts

The archive contains metrics, predictions, sample IDs, logs and figures.
Large checkpoints and the fitted joblib baseline stay in their run folders.
Keep those files separately when reproducing inference.


In [ ]:
EVIDENCE_ZIP = BASE.parent / f"{TAG}_evidence.zip"
with zipfile.ZipFile(EVIDENCE_ZIP, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in BASE.rglob("*"):
        if path.is_file() and "best_model" not in path.parts and path.suffix in {".json", ".csv", ".png", ".md", ".npy"}:
            archive.write(path, path.relative_to(BASE))
print("Evidence archive:", EVIDENCE_ZIP)
print("Checkpoint:", BEST_NEW/"best_model")